In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


def choose_device():
    """选择当前 Kaggle 环境里真正能安全使用的设备。"""
    if not torch.cuda.is_available():
        print("GPU not found, using CPU. 也能跑小实验，但会慢一些。")
        return "cpu"

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print("GPU:", name)
    print("CUDA capability:", f"sm_{props.major}{props.minor}")

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 你遇到的 PyTorch 2.10.0+cu128 构建只支持 sm_70 及以上。
    # 这种情况下 torch.cuda.is_available() 仍然可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print("\n注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建不支持 sm_60。")
        print("本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。")
        print("如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。\n")
        return "cpu"

    return "cuda"


device = choose_device()


# 固定随机种子。
# 训练神经网络时，初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == "cuda":
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对我们这种学习实验来说，可以让矩阵乘法更快。
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print("Using device:", device)
print("Seed:", seed)


# 后续我们会按这个顺序继续写：
# Cell 2：准备一个极小文本数据集，并把字符变成整数 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：先实现 BigramLanguageModel，理解最小语言模型
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本


In [ ]:
# 第 2 个 cell：读取官方 Tiny Shakespeare，并把文本变成字符级 token
#
# 这一格对应 nanoGPT 官方源码：
#   nanoGPT-reference/data/shakespeare_char/prepare.py
#
# 官方 prepare.py 做了几件事：
# 1. 下载 tinyshakespeare/input.txt
# 2. 找出文本里出现过的所有字符
# 3. 给每个字符分配一个整数编号
# 4. 把整篇文本编码成一长串整数
# 5. 切成 train.bin 和 val.bin
#
# 我们这里先不急着保存 .bin 文件，而是把每一步都摊开看清楚。

from pathlib import Path


def find_tiny_shakespeare_file():
    """在 Kaggle input 或本地参考目录中寻找官方 tinyshakespeare 的 input.txt。"""
    candidate_paths = []

    # Kaggle 上传数据集后，通常会出现在 /kaggle/input/某个数据集名字/input.txt。
    kaggle_input_dir = Path("/kaggle/input")
    if kaggle_input_dir.exists():
        candidate_paths.extend(sorted(kaggle_input_dir.rglob("input.txt")))

    # 本地调试时，如果你手动放到了官方目录，也能被找到。
    candidate_paths.append(Path("nanoGPT-reference/data/shakespeare_char/input.txt"))

    for path in candidate_paths:
        if path.exists() and path.is_file():
            return path

    searched = [
        "/kaggle/input/**/input.txt",
        "nanoGPT-reference/data/shakespeare_char/input.txt",
    ]
    raise FileNotFoundError(
        "没有找到 tinyshakespeare 的 input.txt。\n"
        "请确认你已经把官方 input.txt 上传到 Kaggle 的 input 里。\n"
        "代码搜索过的位置：\n- " + "\n- ".join(searched)
    )


input_path = find_tiny_shakespeare_file()
text = input_path.read_text(encoding="utf-8")

print("找到数据文件:", input_path)
print("文本总字符数:", f"{len(text):,}")
print("\n前 500 个字符预览：")
print(text[:500])


# 什么是“字符级 token”？
# 在这一版最小 nanoGPT 里，我们先采用最容易理解的做法：
# 一个字符就是一个 token。
#
# 例如英文里的 'a'、'b'、空格、换行符，都各自算一个 token。
# 模型看不到真正的文字，它只能看到数字，所以我们需要：
#   字符 -> 整数编号
# 这个映射表就是最简单的 tokenizer。

chars = sorted(list(set(text)))
vocab_size = len(chars)

print("\n文本里一共出现了多少种不同字符:", vocab_size)
print("这些字符组成的词表如下：")
print("".join(chars))


# stoi = string to integer，意思是“字符到整数”
# itos = integer to string，意思是“整数到字符”
#
# 举个例子：
#   如果 stoi['a'] = 39
#   就表示字符 'a' 被我们编号成 39。
#   这个编号本身没有语义，不代表 a 比 b 更大，也不代表第 39 个字符更重要。
#   它只是一个身份证号码，方便模型用数字处理文字。

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s):
    """把字符串变成整数列表。"""
    return [stoi[ch] for ch in s]


def decode(ids):
    """把整数列表还原成字符串。"""
    return "".join(itos[i] for i in ids)


print("\n前 20 个字符的编号示例：")
for ch in chars[:20]:
    readable = ch
    if ch == "\n":
        readable = "\\n"
    elif ch == " ":
        readable = "空格"
    print(f"{readable!r} -> {stoi[ch]}")


example_text = "First Citizen:"
example_ids = encode(example_text)

print("\n编码/解码小例子：")
print("原始文本:", example_text)
print("编码后:", example_ids)
print("解码回去:", decode(example_ids))


# 现在把整篇莎士比亚文本都变成数字。
# 这一步之后，原来的一大段英文会变成一条很长很长的整数序列。
# 后面的模型训练，本质上就是让模型看前面一段整数，猜下一个整数。

data = torch.tensor(encode(text), dtype=torch.long)
print("\n整篇文本编码后的张量形状:", data.shape)
print("前 100 个 token 编号:")
print(data[:100].tolist())
print("把这 100 个 token 解码回文字:")
print(decode(data[:100].tolist()))


# 按官方 shakespeare_char/prepare.py 的做法：
# 前 90% 作为训练集，后 10% 作为验证集。
#
# 训练集：给模型反复练习用，相当于教材例题。
# 验证集：训练时不直接拿来背，用来检查模型是不是只会死记硬背教材。

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("\n训练集 token 数:", f"{len(train_data):,}")
print("验证集 token 数:", f"{len(val_data):,}")
print("训练集占比:", f"{len(train_data) / len(data):.1%}")
print("验证集占比:", f"{len(val_data) / len(data):.1%}")


# 这一格产生的关键变量：
#   text       原始字符串
#   chars      词表里的所有字符
#   vocab_size 词表大小，也就是模型最终可能输出多少种字符
#   stoi       字符 -> 编号
#   itos       编号 -> 字符
#   encode     编码函数
#   decode     解码函数
#   train_data 训练用 token 序列
#   val_data   验证用 token 序列
#
# 下一格 Cell 3 会讲：
#   怎么从 train_data 中切出一小段 x，
#   再让 y 变成“x 的每个位置往后挪一位”的答案。
